# Lab 5 · Árboles y grafos con datos reales

**Estructura de Datos · UTP · ED2026** · Duración: 60 minutos · Trabajo individual

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/chriscard04/ed2026-lab5/blob/estable/notebooks/Lab5_Arboles_Grafos.ipynb)

Hoy tu sector tuvo un mal día: **se fue la luz** y, además, **cerraron una estación del Metro**.
Vas a usar árboles y grafos para entender qué pasó y decidir qué hacer.

Tus datos están en una base de datos en la nube y **son solo tuyos**: otra persona tiene otra red,
otra ciudad y otras respuestas. Por eso no sirve copiar resultados.

| Min | Misión | Estructura | Recorridos |
|---|---|---|---|
| 0–7 | **M0 · Conéctate a tus datos** | API REST, registros | paginación |
| 7–30 | **M1 · Se fue la luz** | árbol | DFS (pila), BFS (cola), heap |
| 30–52 | **M2 · Moverse por tu ciudad** | grafo con pesos | BFS, DFS, Dijkstra (heap) |
| 52–60 | **Informe** | — | elegir la frontera correcta |

### Cómo se trabaja

1. **No escribes código desde cero.** Cada bloque trae el código completo y comentado.
   Léelo y ejecútalo (▶ o `Shift + Enter`).
2. Donde veas **`✏️ AJUSTA`** y **`???`** tienes que cambiar algo pequeño (por ejemplo, elegir
   `CustomPila()` o `CustomCola()`). Si ejecutas sin cambiarlo, Python marca `SyntaxError`: es a propósito.
3. Las respuestas se envían con `lab.responder("ID", valor)`. Cada celda de respuesta te dice el
   **FORMATO** exacto y un **EJEMPLO**. Si el formato está mal, no se envía y **no gastas intento**.
4. Tienes 3 intentos por pregunta. Si fallas, recibes una pista. Los análisis los revisa Claude
   (2 revisiones por pregunta) y **la nota final la pone el docente**.
5. En cualquier momento puedes ver tu avance con `lab.estado()`.

Referencia de la API y de los formatos: [API docs del laboratorio](https://github.com/chriscard04/ed2026-lab5/tree/estable/docs).

## Preparación

**Antes de ejecutar:** abre el ícono de la **llave** (Secrets) en la barra izquierda de Colab y crea
dos secretos con el interruptor *Acceso del notebook* activado:

| Nombre | Valor |
|---|---|
| `LAB_EMAIL` | tu correo institucional |
| `LAB_PASSWORD` | la clave temporal que te dio el docente |

Así tu clave nunca queda escrita en el código.

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# CELDA 0 · Preparar el laboratorio (solo ejecútala, no cambies nada)
#   1. Instala 'labkit', la caja de herramientas del curso.
#   2. Inicia tu sesión con tus credenciales guardadas en Secrets.
# ─────────────────────────────────────────────────────────────────────
import importlib.util
if importlib.util.find_spec("labkit") is None:
    !pip install -q "git+https://github.com/chriscard04/ed2026-lab5@estable#subdirectory=labkit"

import heapq
from labkit import Lab, CustomPila, CustomCola, FronteraHeap, nombre_frontera, camino

lab = Lab.login()

In [ ]:
# Tu asignación personal: lo que te tocó a TI (otra persona tiene otros valores).
asignacion = lab.asignacion()
for clave, valor in asignacion.items():
    print(f"{clave:<22} {valor}")

---
## Misión 0 · Conéctate a tus datos (7 min)

Casi todas las aplicaciones actuales consumen datos de una **API REST**: piden datos con una URL
y reciben **JSON**. Tus datos están en tablas de PostgreSQL (Supabase). Cada fila llega como un
**diccionario** de Python, igual que los registros que ya conoces.

Tu sesión lleva un **token** que identifica quién eres. La base de datos aplica una regla (RLS)
que solo te deja ver **tus** filas.

### 0.1 Tu primera petición

**Para qué sirve:** así lee datos cualquier app: una URL, unos parámetros y una respuesta JSON.

**Qué vas a observar:** el código de estado HTTP (200 = todo bien) y que cada fila es un diccionario.

**Concepto:** `GET /rest/v1/grid_nodes?select=id,parent_id,kind,name&limit=3`

In [ ]:
# lab.get(tabla, parametros) hace la petición GET con tu token.
#   select → qué columnas quieres
#   order  → en qué orden
#   limit  → cuántas filas como máximo
respuesta = lab.get("grid_nodes", {"select": "id,parent_id,kind,name,demand_kw", "order": "id", "limit": 3})

print("Código HTTP:", respuesta.status_code)     # 200 = OK
for fila in respuesta.json():                     # .json() convierte el JSON en una lista de diccionarios
    print(fila)

### 0.2 La trampa de las 1000 filas

**Para qué sirve:** los servidores limitan cuántas filas devuelven por petición para no saturarse.

**Qué vas a observar:** que pides TODO y llegan menos filas de las que tiene tu red.

In [ ]:
# Pedimos todas las filas, sin límite... ¿llegan todas?
todo = lab.get("grid_nodes", {"select": "*"}).json()
filas_sin_paginar = len(todo)
print("Filas recibidas:", filas_sin_paginar)

In [ ]:
# ══════════════════════════════════════════════════════════════════
# RESPONDE · M0.1 · Filas sin paginar
#   FORMATO : Número entero, sin comillas, sin puntos ni comas de miles, sin decimales y sin unidades.
#   EJEMPLO : lab.responder("M0.1", 250)
#   NO      : "1482"   1,482   1482.0
# ══════════════════════════════════════════════════════════════════
lab.responder("M0.1", filas_sin_paginar)

### 0.3 Paginar: pedir de 1000 en 1000

**Para qué sirve:** así se descargan tablas grandes de cualquier API (Supabase, GitHub, redes sociales...).

**Qué vas a observar:** cuántas peticiones hacen falta y cómo se detiene el ciclo.

**Concepto:** `limit` (tamaño de página) y `offset` (desde qué fila empezar).

In [ ]:
LIMITE = ???   # ✏️ AJUSTA: el tope de filas por respuesta que descubriste en 0.2

def descargar_todo(tabla, orden):
    filas = []          # aquí juntamos todos los lotes
    offset = 0          # desde qué fila pedimos
    peticiones = 0
    while True:
        respuesta = lab.get(tabla, {"select": "*", "order": orden, "limit": LIMITE, "offset": offset})
        lote = respuesta.json()
        peticiones += 1
        print(f"Petición {peticiones}: offset={offset} → {len(lote)} filas")
        filas.extend(lote)
        if len(lote) < LIMITE:     # llegó una página incompleta: era la última
            break
        offset += LIMITE           # la próxima página empieza donde terminó esta
    return filas, peticiones

nodos, peticiones = descargar_todo("grid_nodes", "id")
print("Total de nodos de tu red:", len(nodos))

lab.nodos_conocidos = {n["id"] for n in nodos}   # labkit te avisará si escribes un id que no existe
lab.anotar("0.3", limite=LIMITE)

**Qué deberías ver:** 2 o 3 peticiones y un total mayor que 1000.

**Si algo falla:**
- `SyntaxError` → todavía tienes `???`.
- El total es 1000 → tu `LIMITE` es mayor que el tope del servidor: el ciclo cree que ya terminó.

In [ ]:
# ══════════════════════════════════════════════════════════════════
# RESPONDE · M0.2 · Total de nodos de tu red
#   FORMATO : Número entero, sin comillas, sin puntos ni comas de miles, sin decimales y sin unidades.
#   EJEMPLO : lab.responder("M0.2", 1693)
#   NO      : "1482"   1,482   1482.0
# ══════════════════════════════════════════════════════════════════
lab.responder("M0.2", len(nodos))

In [ ]:
# ══════════════════════════════════════════════════════════════════
# RESPONDE · M0.3 · Peticiones de la paginación
#   FORMATO : Número entero, sin comillas, sin puntos ni comas de miles, sin decimales y sin unidades.
#   EJEMPLO : lab.responder("M0.3", 7)
#   NO      : "1482"   1,482   1482.0
# ══════════════════════════════════════════════════════════════════
lab.responder("M0.3", peticiones)

### 0.4 De filas a árbol

**Para qué sirve:** convertir registros en una estructura que se pueda recorrer.

**Qué vas a observar:** dos diccionarios: `info` (id → fila) e `hijos` (id → lista de hijos en orden alfabético).

In [ ]:
info = {n["id"]: n for n in nodos}          # acceso directo a cada nodo por su id

hijos = {n["id"]: [] for n in nodos}        # cada nodo empieza sin hijos
for n in nodos:
    if n["parent_id"] is not None:          # la subestación (raíz) no tiene padre
        hijos[n["parent_id"]].append(n["id"])
for u in hijos:
    hijos[u].sort()                         # convención del curso: vecinos en orden alfabético

from collections import Counter
print(Counter(n["kind"] for n in nodos))
print("Hijos de la subestación:", hijos["SE"])

---
## Misión 1 · Se fue la luz (23 min)

La red eléctrica de tu sector **es un árbol**: la subestación (raíz) alimenta circuitos, los
circuitos alimentan ramales, los ramales alimentan transformadores y estos a los clientes. Cada
nodo tiene un solo padre: si un nodo falla, **todo lo que cuelga de él se queda sin luz**.

Además hay un **circuito rural**: más de 1200 postes en serie, uno detrás de otro.

### 1.1 El esqueleto único: frontera + visitados

**Para qué sirve:** es la MISMA función para DFS, BFS y Dijkstra. Solo cambia la frontera.

**Qué vas a observar:** que la función no sabe si recorre un árbol o un grafo: solo saca de la frontera y agrega vecinos.

**Concepto:** `CustomPila()` → DFS · `CustomCola()` → BFS · `FronteraHeap()` → Dijkstra.

In [ ]:
def recorrer(vecinos, inicio, frontera, minutos=None, destino=None):
    padre = {}      # padre[nodo] = desde dónde llegué (también sirve como "visitados")
    costo = {}      # costo[nodo] = costo acumulado (niveles o minutos)
    orden = []      # en qué orden visité los nodos

    frontera.agregar(inicio, padre=None, costo=0)
    while not frontera.esta_vacia():
        nodo, de_donde, acumulado = frontera.sacar()     # ← la frontera decide QUIÉN sale
        if nodo in padre:                                # ya lo visité: lo descarto
            continue
        padre[nodo] = de_donde
        costo[nodo] = acumulado
        orden.append(nodo)
        if nodo == destino:                              # si busco un destino, paro al encontrarlo
            break
        for vecino in vecinos.get(nodo, []):
            if vecino not in padre:
                paso = minutos[(nodo, vecino)] if minutos else 1    # sin minutos, cada paso vale 1
                frontera.agregar(vecino, padre=nodo, costo=acumulado + paso)
    return orden, padre, costo

print("Listo: recorrer() está definida.")

### 1.2 Predice antes de ejecutar

**Para qué sirve:** predecir te obliga a entender el comportamiento, no solo a ver el resultado.

**Qué vas a observar:** tu ramal de práctica: un pedazo pequeño de tu red.

In [ ]:
RAMAL = asignacion["ramal_practica"]
K = asignacion["prediccion_bfs_k"]
lab.ver_subarbol(hijos, info, RAMAL)
print(f"\nTu k para la predicción BFS es: {K}")

Responde **sin ejecutar ningún recorrido todavía** (usa papel si te ayuda). Recuerda: los hijos se
agregan a la frontera en **orden alfabético de id**.

- **M1.P1:** con BFS (cola) desde tu ramal, ¿cuál es el nodo número **k** que SALE de la cola?
- **M1.P2:** con DFS (pila) desde tu ramal, ¿qué hay en la pila (de **fondo a tope**) justo después de
  sacar el **segundo** nodo y agregar sus hijos?

In [ ]:
# ══════════════════════════════════════════════════════════════════
# RESPONDE · M1.P1 · Predicción BFS en el ramal
#   FORMATO : Texto entre comillas con el id EXACTO del nodo, como aparece en tus datos (mayúsculas, guion y ceros).
#   EJEMPLO : lab.responder("M1.P1", "K-0002")
#   NO      : "k-0002"   "K-2"   "Cliente 0002"
# ══════════════════════════════════════════════════════════════════
lab.responder("M1.P1", "???")

In [ ]:
# ══════════════════════════════════════════════════════════════════
# RESPONDE · M1.P2 · Predicción de la pila
#   FORMATO : Lista entre corchetes [ ] con ids de nodo entre comillas, separados por comas, en el orden que pide la pregunta.
#   EJEMPLO : lab.responder("M1.P2", ["T1-01", "K-0004", "K-0005"])
#   NO      : "T1-01, K-0004"   ("T1-01", "K-0004")   ["t1-01"]
# ══════════════════════════════════════════════════════════════════
lab.responder("M1.P2", ["???", "???"])

### 1.3 Ahora sí: DFS paso a paso

**Para qué sirve:** comprobar tu predicción viendo la pila por dentro.

**Qué vas a observar:** que la pila saca el ÚLTIMO que entró, así que baja primero por el hijo de id mayor.

In [ ]:
frontera = ???          # ✏️ AJUSTA: queremos DFS (mira la tabla del bloque 1.1)
orden_ramal, _, _ = recorrer(hijos, RAMAL, frontera)

lab.ver_frontera(frontera)       # tabla: qué salió, qué se agregó y cómo quedó la frontera
print("\nOrden de visita:", orden_ramal)
lab.anotar("1.3", frontera=nombre_frontera(frontera))

**Qué deberías ver:** una tabla donde la columna *Frontera* se lee de fondo a tope; compárala con tu predicción M1.P2.

**Si algo falla:**
- El orden sale por niveles (todos los transformadores primero) → usaste `CustomCola()`.

In [ ]:
# ══════════════════════════════════════════════════════════════════
# RESPONDE · M1.T1 · Traza DFS del ramal
#   FORMATO : Lista entre corchetes [ ] con ids de nodo entre comillas, separados por comas, en el orden que pide la pregunta.
#   EJEMPLO : lab.responder("M1.T1", ["R1-1", "T1-02", "K-0006", "K-0005", "K-0004", "T1-01", "K-0003", "K-0002"])
#   NO      : "T1-01, K-0004"   ("T1-01", "K-0004")   ["t1-01"]
# ══════════════════════════════════════════════════════════════════
lab.responder("M1.T1", orden_ramal)

In [ ]:
# Comparación: el MISMO ramal con BFS (cola). Úsalo para revisar tu predicción M1.P1.
cola = CustomCola()
recorrer(hijos, RAMAL, cola)
lab.ver_frontera(cola)

### 1.4 La recursión no aguanta tu red

**Para qué sirve:** en sistemas reales hay estructuras muy profundas (carpetas, comentarios, líneas eléctricas).

**Qué vas a observar:** qué pasa al calcular la carga con la versión recursiva clásica.

In [ ]:
def carga_recursiva(u):
    # carga de u = su demanda + la carga de cada hijo (cada llamada espera a sus hijos)
    return (info[u]["demand_kw"] or 0) + sum(carga_recursiva(h) for h in hijos[u])

try:
    print("Carga total:", carga_recursiva("SE"))
except RecursionError as error:
    print("RecursionError:", error)
    print("Python permite ~1000 llamadas anidadas y tu circuito rural tiene más de 1200 postes en serie.")

### 1.5 Carga de cada nodo sin recursión

**Para qué sirve:** así calcula Windows el tamaño de una carpeta, o la empresa eléctrica la carga de un circuito.

**Qué vas a observar:** que leyendo AL REVÉS el orden de la pila, cada hijo suma su carga al padre antes que el padre suba la suya.

**Concepto:** DFS iterativo con pila · orden inverso = postorden.

In [ ]:
# PASO 1. Recorremos TODA la red con DFS, usando una pila en vez de recursión.
frontera = ???          # ✏️ AJUSTA: DFS
orden, _, _ = recorrer(hijos, "SE", frontera)

# PASO 2. Cada nodo empieza con su propia demanda (solo los clientes tienen demanda).
carga = {u: info[u]["demand_kw"] or 0 for u in orden}

# PASO 3. Al revés: cada nodo le pasa su carga acumulada a su padre.
for u in reversed(orden):
    padre = info[u]["parent_id"]
    if padre is not None:
        carga[padre] += carga[u]

print("Carga total del sector:", carga["SE"], "kW")
lab.ver_cargas(carga, info, tipo="circuito")
lab.anotar("1.5", frontera=nombre_frontera(frontera))

**Qué deberías ver:** la carga total y un gráfico con la carga de cada circuito.

**Si algo falla:**
- `SyntaxError` → todavía tienes `???`.
- Con `CustomCola()` también funciona. ¿Por qué? Piénsalo para el análisis M1.A1.

In [ ]:
# ══════════════════════════════════════════════════════════════════
# RESPONDE · M1.3 · Carga total del sector
#   FORMATO : Número entero, sin comillas, sin puntos ni comas de miles, sin decimales y sin unidades.
#   EJEMPLO : lab.responder("M1.3", 2453)
#   NO      : "1482"   1,482   1482.0
# ══════════════════════════════════════════════════════════════════
lab.responder("M1.3", carga["SE"])

In [ ]:
# Reto sorpresa: el servidor elige un nodo de TU red y tú respondes su carga.
nodo = lab.reto("M1.2")
print("Carga de", nodo, "=", carga[nodo], "kW")

In [ ]:
# ══════════════════════════════════════════════════════════════════
# RESPONDE · M1.2 · Reto sorpresa: carga de un nodo
#   FORMATO : Número entero, sin comillas, sin puntos ni comas de miles, sin decimales y sin unidades.
#   EJEMPLO : lab.responder("M1.2", 2453)
#   NO      : "1482"   1,482   1482.0
#   OJO     : primero pide tu reto con lab.reto("M1.2"); cada reto sirve para un envío.
# ══════════════════════════════════════════════════════════════════
lab.responder("M1.2", carga[nodo])

### 1.6 La falla: ¿quién se quedó sin luz?

**Para qué sirve:** es lo primero que necesita saber el centro de control al recibir una alarma.

**Qué vas a observar:** que todo el subárbol del nodo fallado se apaga.

In [ ]:
FALLA = asignacion["falla"]["nodo"]
print("Falla:", FALLA, "·", asignacion["falla"]["descripcion"])

subarbol, _, _ = recorrer(hijos, FALLA, CustomPila())     # todo lo que cuelga de la falla

# ✏️ AJUSTA: queremos contar solo los nodos de tipo cliente (mira la columna kind)
afectados = [u for u in subarbol if info[u]["kind"] == "???"]
criticos_afectados = [u for u in afectados if info[u]["critical"] is not None]

print("Clientes sin luz:", len(afectados))
print("Carga perdida:", carga[FALLA], "kW")
for u in criticos_afectados:
    print("  CRÍTICO:", u, info[u]["name"])

In [ ]:
# ══════════════════════════════════════════════════════════════════
# RESPONDE · M1.4 · Clientes sin luz
#   FORMATO : Número entero, sin comillas, sin puntos ni comas de miles, sin decimales y sin unidades.
#   EJEMPLO : lab.responder("M1.4", 18)
#   NO      : "1482"   1,482   1482.0
# ══════════════════════════════════════════════════════════════════
lab.responder("M1.4", len(afectados))

In [ ]:
# ══════════════════════════════════════════════════════════════════
# RESPONDE · M1.5 · Clientes críticos afectados
#   FORMATO : Lista entre corchetes [ ] con ids de nodo entre comillas, separados por comas, en el orden que pide la pregunta.
#   EJEMPLO : lab.responder("M1.5", ["K-0379"])
#   NO      : "T1-01, K-0004"   ("T1-01", "K-0004")   ["t1-01"]
#   OJO     : el orden de la lista no importa.
# ══════════════════════════════════════════════════════════════════
lab.responder("M1.5", criticos_afectados)

### 1.7 BFS: lo urgente más cerca

**Para qué sirve:** una cuadrilla quiere revisar primero lo más cercano a la subestación.

**Qué vas a observar:** en cuántos pasos encuentra BFS el primer cliente crítico, comparado con DFS.

**Concepto:** BFS avanza por niveles; `costo[nodo]` es el nivel (número de aristas desde la raíz).

In [ ]:
frontera = ???          # ✏️ AJUSTA: BFS
orden_bfs, _, nivel = recorrer(hijos, "SE", frontera)

def primer_critico(orden):
    for paso, u in enumerate(orden, start=1):
        if info[u]["critical"] is not None:
            return u, paso

critico_bfs, pasos_bfs = primer_critico(orden_bfs)
critico_dfs, pasos_dfs = primer_critico(recorrer(hijos, "SE", CustomPila())[0])

print(f"BFS: {critico_bfs} ({info[critico_bfs]['name']}) en el paso {pasos_bfs}, nivel {nivel[critico_bfs]}")
print(f"DFS: {critico_dfs} ({info[critico_dfs]['name']}) en el paso {pasos_dfs}")
print("Altura de tu red:", max(nivel.values()))
lab.anotar("1.7", frontera=nombre_frontera(frontera))

In [ ]:
# ══════════════════════════════════════════════════════════════════
# RESPONDE · M1.6 · Primer cliente crítico que encuentra BFS
#   FORMATO : Texto entre comillas con el id EXACTO del nodo, como aparece en tus datos (mayúsculas, guion y ceros).
#   EJEMPLO : lab.responder("M1.6", "K-0001")
#   NO      : "k-0002"   "K-2"   "Cliente 0002"
# ══════════════════════════════════════════════════════════════════
lab.responder("M1.6", critico_bfs)

In [ ]:
# ══════════════════════════════════════════════════════════════════
# RESPONDE · M1.7 · Altura de la red
#   FORMATO : Número entero, sin comillas, sin puntos ni comas de miles, sin decimales y sin unidades.
#   EJEMPLO : lab.responder("M1.7", 1257)
#   NO      : "1482"   1,482   1482.0
#   NOTA    : pregunta opcional.
# ══════════════════════════════════════════════════════════════════
lab.responder("M1.7", max(nivel.values()))

### 1.8 El heap por dentro

**Para qué sirve:** un heap entrega siempre el mínimo en O(1) y saca o mete elementos en O(log n).

**Qué vas a observar:** cómo se guarda en un arreglo y qué pasa al sacar la raíz.

**Concepto:** min-heap: padre ≤ hijos · hijos de i: 2i+1 y 2i+2.

In [ ]:
# Estos son los % de carga de 5 de tus transformadores, metidos con heappush en este orden:
valores = asignacion["heap_valores"]
heap_demo = []
for v in valores:
    heapq.heappush(heap_demo, v)

print("Metidos en este orden:", valores)
lab.ver_heap(heap_demo)

Responde **antes** de ejecutar el `heappop` de la celda siguiente.

In [ ]:
# ══════════════════════════════════════════════════════════════════
# RESPONDE · M1.P3 · Heap: nueva raíz
#   FORMATO : Número entero, sin comillas, sin puntos ni comas de miles, sin decimales y sin unidades.
#   EJEMPLO : lab.responder("M1.P3", 81)
#   NO      : "1482"   1,482   1482.0
# ══════════════════════════════════════════════════════════════════
lab.responder("M1.P3", "???")

In [ ]:
# ══════════════════════════════════════════════════════════════════
# RESPONDE · M1.P4 · Heap: arreglo después de sacar
#   FORMATO : Lista entre corchetes [ ] con números enteros separados por comas, en el orden del arreglo.
#   EJEMPLO : lab.responder("M1.P4", [81, 126, 111, 140])
#   NO      : "81, 126"   ["81", "126"]   (81, 126)
# ══════════════════════════════════════════════════════════════════
lab.responder("M1.P4", ["???"])

In [ ]:
# Ahora sí: sacamos la raíz y vemos cómo quedó (compara con tus predicciones).
salio = heapq.heappop(heap_demo)
print("Salió:", salio)
lab.ver_heap(heap_demo)

### 1.9 Los transformadores más sobrecargados

**Para qué sirve:** decidir qué equipos reemplazar primero sin ordenar toda la red.

**Qué vas a observar:** que un heap de MÍNIMOS de tamaño K deja al final los K MAYORES.

In [ ]:
transformadores = [u for u in info if info[u]["kind"] == "transformador"]
porcentaje = {t: round(100 * carga[t] / info[t]["capacity_kw"]) for t in transformadores}

K = ???    # ✏️ AJUSTA: usa el top_k de tu asignación
heap = []
for t in transformadores:
    heapq.heappush(heap, (porcentaje[t], t))   # se ordena por % de carga
    if len(heap) > K:
        heapq.heappop(heap)                    # sale el MENOR: el menos sobrecargado del grupo

top = sorted(heap, reverse=True)              # de mayor a menor
for pct, t in top:
    print(f"{t}: {pct} % de su capacidad ({carga[t]} kW / {info[t]['capacity_kw']} kW)")
lab.ver_cargas(carga, info, tipo="transformador", top=10, porcentaje=porcentaje)

In [ ]:
# ══════════════════════════════════════════════════════════════════
# RESPONDE · M1.8 · Top de transformadores sobrecargados
#   FORMATO : Lista entre corchetes [ ] con ids de nodo entre comillas, separados por comas, en el orden que pide la pregunta.
#   EJEMPLO : lab.responder("M1.8", ["T4-04", "T4-01", "T3-05", "T1-02", "T4-06"])
#   NO      : "T1-01, K-0004"   ("T1-01", "K-0004")   ["t1-01"]
# ══════════════════════════════════════════════════════════════════
lab.responder("M1.8", [t for pct, t in top])

### Análisis de la Misión 1

Escribe con tus palabras y **con tus datos** (entre 40 y 600 caracteres). Claude te devolverá una
retroalimentación y un nivel sugerido; puedes mejorar el texto y enviarlo una segunda vez.

In [ ]:
# ══════════════════════════════════════════════════════════════════
# RESPONDE · M1.A1 · ¿Por qué funciona el orden al revés?
#   FORMATO : Texto entre comillas de 40 a 600 caracteres, en oraciones completas. Usa los datos de TU caso.
#   EJEMPLO : lab.responder("M1.A1", "En mi red, la falla en ... deja ... clientes sin luz porque ...")
#   NO      : ""   "porque sí"   un texto de más de 600 caracteres
# ══════════════════════════════════════════════════════════════════
lab.responder("M1.A1", "Escribe aquí tu análisis.")

In [ ]:
# ══════════════════════════════════════════════════════════════════
# RESPONDE · M1.A2 · BFS frente a DFS para encontrar lo urgente
#   FORMATO : Texto entre comillas de 40 a 600 caracteres, en oraciones completas. Usa los datos de TU caso.
#   EJEMPLO : lab.responder("M1.A2", "En mi red, la falla en ... deja ... clientes sin luz porque ...")
#   NO      : ""   "porque sí"   un texto de más de 600 caracteres
# ══════════════════════════════════════════════════════════════════
lab.responder("M1.A2", "Escribe aquí tu análisis.")

In [ ]:
# ══════════════════════════════════════════════════════════════════
# RESPONDE · M1.A3 · Heap de mínimos para buscar los mayores
#   FORMATO : Texto entre comillas de 40 a 600 caracteres, en oraciones completas. Usa los datos de TU caso.
#   EJEMPLO : lab.responder("M1.A3", "En mi red, la falla en ... deja ... clientes sin luz porque ...")
#   NO      : ""   "porque sí"   un texto de más de 600 caracteres
# ══════════════════════════════════════════════════════════════════
lab.responder("M1.A3", "Escribe aquí tu análisis.")

In [ ]:
# ══════════════════════════════════════════════════════════════════
# RESPONDE · M1.A4 · ¿A dónde mandas la primera cuadrilla?
#   FORMATO : Texto entre comillas de 40 a 600 caracteres, en oraciones completas. Usa los datos de TU caso.
#   EJEMPLO : lab.responder("M1.A4", "En mi red, la falla en ... deja ... clientes sin luz porque ...")
#   NO      : ""   "porque sí"   un texto de más de 600 caracteres
# ══════════════════════════════════════════════════════════════════
lab.responder("M1.A4", "Escribe aquí tu análisis.")

---
## Misión 2 · Moverse por tu ciudad (22 min)

Ahora tu red es un **grafo con pesos**: estaciones del Metro (L1 y L2) y paradas de bus unidas por
tramos que tardan cierta cantidad de **minutos**. Las aristas no tienen dirección (se viaja en ambos
sentidos) y hay ciclos. Además, **una estación está cerrada** por mantenimiento.

*Red simplificada: los nombres de estaciones son reales; los tiempos y las rutas de bus son simulados.*

### 2.1 Tu ciudad desde la API

**Para qué sirve:** armar un grafo a partir de dos tablas: estaciones (vértices) y tramos (aristas).

**Qué vas a observar:** que la estación cerrada no debe entrar al grafo.

**Concepto:** lista de adyacencia: diccionario estación → vecinos.

In [ ]:
estaciones = lab.descargar("stations")     # ya conoces la paginación: labkit la hace por ti
tramos = lab.descargar("segments")

# ✏️ AJUSTA: solo deben entrar al grafo las estaciones con este estado (mira la columna status)
activas = {e["code"] for e in estaciones if e["status"] == "???"}

vecinos = {c: [] for c in activas}
minutos = {}
for t in tramos:
    a, b = t["from_code"], t["to_code"]
    if a in activas and b in activas:          # si un extremo está cerrado, el tramo no sirve
        vecinos[a].append(b)                   # no dirigido: se agrega en los dos sentidos
        vecinos[b].append(a)
        minutos[(a, b)] = minutos[(b, a)] = t["minutes"]
for c in vecinos:
    vecinos[c].sort()                          # convención: vecinos en orden alfabético

nombre = {e["code"]: e["name"] for e in estaciones}
print("Estación cerrada:", asignacion["estacion_cerrada"], "·", nombre[asignacion["estacion_cerrada"]])
print("Estaciones activas:", len(activas))
lab.ver_mapa(estaciones, tramos)

### 2.2 Predice: BFS en tu ciudad

**Para qué sirve:** practicar el orden por niveles en un grafo real.

**Qué vas a observar:** que BFS saca primero las estaciones a 1 parada, luego a 2, etc.

In [ ]:
par1 = asignacion["par1"]
ORIGEN, DESTINO = par1["origen"], par1["destino"]
K2 = asignacion["prediccion_bfs_k_m2"]
print(f"Par 1: {ORIGEN} ({nombre[ORIGEN]}) → {DESTINO} ({nombre[DESTINO]})")
print(f"Vecinos de {ORIGEN}: {vecinos[ORIGEN]}")
for v in vecinos[ORIGEN]:
    print(f"  vecinos de {v}: {vecinos[v]}")
print(f"\nTu k para la predicción es: {K2}")

In [ ]:
# ══════════════════════════════════════════════════════════════════
# RESPONDE · M2.P1 · Predicción BFS en tu ciudad
#   FORMATO : Texto entre comillas con el código de la estación: 3 letras MAYÚSCULAS (Metro) o B#-## (parada de bus).
#   EJEMPLO : lab.responder("M2.P1", "FDC")
#   NO      : "alb"   "Albrook"   " ALB"
# ══════════════════════════════════════════════════════════════════
lab.responder("M2.P1", "???")

### 2.3 Tres fronteras, una misma pregunta

**Para qué sirve:** elegir la herramienta correcta: menos paradas no siempre es más rápido.

**Qué vas a observar:** tres rutas distintas para el mismo par en el mapa.

**Concepto:** BFS minimiza paradas · Dijkstra minimiza minutos · DFS solo encuentra *una* ruta.

In [ ]:
def minutos_de(ruta):
    return sum(minutos[(a, b)] for a, b in zip(ruta, ruta[1:]))

# BFS: menos paradas
_, padre_bfs, _ = recorrer(vecinos, ORIGEN, ???, destino=DESTINO)     # ✏️ AJUSTA: BFS
ruta_bfs = camino(padre_bfs, DESTINO)

# Dijkstra: menos minutos. El heap necesita los pesos para sumar minutos.
heap = FronteraHeap()
_, padre_dij, costo_dij = recorrer(vecinos, ORIGEN, heap, minutos=???, destino=DESTINO)   # ✏️ AJUSTA
ruta_dij = camino(padre_dij, DESTINO)

# DFS: alguna ruta
_, padre_dfs, _ = recorrer(vecinos, ORIGEN, CustomPila(), destino=DESTINO)
ruta_dfs = camino(padre_dfs, DESTINO)

for etiqueta, ruta in [("BFS", ruta_bfs), ("Dijkstra", ruta_dij), ("DFS", ruta_dfs)]:
    print(f"{etiqueta:<9} {len(ruta) - 1:>2} paradas · {minutos_de(ruta):>3} min · {' → '.join(ruta)}")
print(f"\n{DESTINO} entró {heap.veces_agregado[DESTINO]} veces al heap y se procesó 1 vez.")
lab.ver_mapa(estaciones, tramos, rutas={"BFS": ruta_bfs, "Dijkstra": ruta_dij, "DFS": ruta_dfs})

**Qué deberías ver:** BFS con menos paradas pero más minutos que Dijkstra, y DFS con una ruta larga.

**Si algo falla:**
- Dijkstra da lo mismo que BFS → no le pasaste `minutos`: cada tramo vale 1.

In [ ]:
# ══════════════════════════════════════════════════════════════════
# RESPONDE · M2.1 · Ruta con menos paradas
#   FORMATO : Lista entre corchetes [ ] con códigos de estación entre comillas, desde el origen hasta el destino.
#   EJEMPLO : lab.responder("M2.1", ["IGC", "STO", "LOT", "B4-01", "B4-02", "PDA", "LAN"])
#   NO      : "ALB, CUR"   ["Albrook", "Curundú"]   ("ALB", "CUR")
# ══════════════════════════════════════════════════════════════════
lab.responder("M2.1", ruta_bfs)

In [ ]:
# ══════════════════════════════════════════════════════════════════
# RESPONDE · M2.2 · Ruta más rápida
#   FORMATO : Lista entre corchetes [ ] con códigos de estación entre comillas, desde el origen hasta el destino.
#   EJEMPLO : lab.responder("M2.2", ["IGC", "VAR", "FDC", "ING", "DOC", "PNU", "SMG", "PDA", "LAN"])
#   NO      : "ALB, CUR"   ["Albrook", "Curundú"]   ("ALB", "CUR")
# ══════════════════════════════════════════════════════════════════
lab.responder("M2.2", ruta_dij)

In [ ]:
# ══════════════════════════════════════════════════════════════════
# RESPONDE · M2.3 · Minutos de la ruta más rápida
#   FORMATO : Número entero, sin comillas, sin puntos ni comas de miles, sin decimales y sin unidades.
#   EJEMPLO : lab.responder("M2.3", 25)
#   NO      : "1482"   1,482   1482.0
# ══════════════════════════════════════════════════════════════════
lab.responder("M2.3", costo_dij[DESTINO])

In [ ]:
# ══════════════════════════════════════════════════════════════════
# RESPONDE · M2.4 · Reingresos al heap
#   FORMATO : Número entero, sin comillas, sin puntos ni comas de miles, sin decimales y sin unidades.
#   EJEMPLO : lab.responder("M2.4", 5)
#   NO      : "1482"   1,482   1482.0
# ══════════════════════════════════════════════════════════════════
lab.responder("M2.4", heap.veces_agregado[DESTINO])

In [ ]:
# Reto sorpresa: el servidor te da otro par; calcula su ruta más rápida.
reto = lab.reto("M2.5")
_, padre_reto, costo_reto = recorrer(vecinos, reto["origen"], FronteraHeap(), minutos=minutos, destino=reto["destino"])
ruta_reto = camino(padre_reto, reto["destino"])
print(ruta_reto, costo_reto[reto["destino"]], "min")

In [ ]:
# ══════════════════════════════════════════════════════════════════
# RESPONDE · M2.5 · Reto sorpresa: ruta más rápida
#   FORMATO : Lista entre corchetes [ ] con códigos de estación entre comillas, desde el origen hasta el destino.
#   EJEMPLO : lab.responder("M2.5", ["ALB", "CUR", "CMY"])
#   NO      : "ALB, CUR"   ["Albrook", "Curundú"]   ("ALB", "CUR")
#   OJO     : primero pide tu reto con lab.reto("M2.5"); cada reto sirve para un envío.
# ══════════════════════════════════════════════════════════════════
lab.responder("M2.5", ruta_reto)

### 2.4 DFS: ¿se partió la ciudad?

**Para qué sirve:** saber si un cierre deja zonas incomunicadas.

**Qué vas a observar:** cuántos grupos de estaciones quedan que no se pueden alcanzar entre sí.

**Concepto:** componente conexa: cada recorrido nuevo desde una estación no visitada encuentra una.

In [ ]:
def contar_componentes(vecinos):
    visitados = set()
    componentes = []
    for estacion in sorted(vecinos):
        if estacion not in visitados:                         # nadie la alcanzó: nueva componente
            orden, _, _ = recorrer(vecinos, estacion, ???)   # ✏️ AJUSTA: DFS
            visitados.update(orden)
            componentes.append(sorted(orden))
    return componentes

componentes = contar_componentes(vecinos)
print("Componentes:", len(componentes))
for i, comp in enumerate(componentes, start=1):
    print(f"  {i}: {len(comp)} estaciones → {comp[:8]}{' ...' if len(comp) > 8 else ''}")

par2 = asignacion["par2"]
print(f"\nPar 2: {par2['origen']} → {par2['destino']}")
print("¿Hay ruta?", par2["destino"] in recorrer(vecinos, par2["origen"], CustomPila())[0])

In [ ]:
# ══════════════════════════════════════════════════════════════════
# RESPONDE · M2.6 · Componentes con la estación cerrada
#   FORMATO : Número entero, sin comillas, sin puntos ni comas de miles, sin decimales y sin unidades.
#   EJEMPLO : lab.responder("M2.6", 4)
#   NO      : "1482"   1,482   1482.0
# ══════════════════════════════════════════════════════════════════
lab.responder("M2.6", len(componentes))

### 2.5 Reconecta tu ciudad

**Para qué sirve:** proponer una obra y comprobar que resuelve el problema ANTES de construirla.

**Qué vas a observar:** que un solo tramo entre las dos componentes deja 1 componente.

In [ ]:
# ✏️ AJUSTA: elige una estación de cada componente (mira la lista de arriba y el mapa)
TRAMO = {"desde": "???", "hasta": "???", "minutos": 6}

if TRAMO["desde"] in vecinos and TRAMO["hasta"] in vecinos:
    prueba = {c: list(vs) for c, vs in vecinos.items()}      # copia del grafo
    prueba[TRAMO["desde"]].append(TRAMO["hasta"])
    prueba[TRAMO["hasta"]].append(TRAMO["desde"])
    print("Componentes con tu tramo:", len(contar_componentes(prueba)))
else:
    print("Cambia los ??? por códigos de estaciones activas.")

In [ ]:
# ══════════════════════════════════════════════════════════════════
# RESPONDE · M2.7 · Tramo que reconecta la ciudad
#   FORMATO : Diccionario con tres claves: "desde" y "hasta" (códigos de estación) y "minutos" (entero de 1 a 30).
#   EJEMPLO : lab.responder("M2.7", {"desde": "PNU", "hasta": "PDA", "minutos": 6})
#   NO      : ["PNU", "PDA", 6]   {"desde": "PNU", "hasta": "PDA", "minutos": "6"}   {"de": "PNU", "a": "PDA"}
# ══════════════════════════════════════════════════════════════════
lab.responder("M2.7", TRAMO)

In [ ]:
# Opcional: radio de 3 paradas con BFS (costo = número de paradas)
ORIGEN_RADIO = asignacion["origen_radio"]
_, _, paradas = recorrer(vecinos, ORIGEN_RADIO, CustomCola())
alcance = [c for c, n in paradas.items() if 1 <= n <= ???]     # ✏️ AJUSTA: radio de tu asignación
print(f"Desde {ORIGEN_RADIO} alcanzas {len(alcance)} estaciones en 3 paradas o menos.")

In [ ]:
# ══════════════════════════════════════════════════════════════════
# RESPONDE · M2.8 · Radio de 3 paradas
#   FORMATO : Número entero, sin comillas, sin puntos ni comas de miles, sin decimales y sin unidades.
#   EJEMPLO : lab.responder("M2.8", 11)
#   NO      : "1482"   1,482   1482.0
#   NOTA    : pregunta opcional.
# ══════════════════════════════════════════════════════════════════
lab.responder("M2.8", len(alcance))

### Análisis de la Misión 2

In [ ]:
# ══════════════════════════════════════════════════════════════════
# RESPONDE · M2.A1 · ¿Qué ruta recomiendas?
#   FORMATO : Texto entre comillas de 40 a 600 caracteres, en oraciones completas. Usa los datos de TU caso.
#   EJEMPLO : lab.responder("M2.A1", "En mi red, la falla en ... deja ... clientes sin luz porque ...")
#   NO      : ""   "porque sí"   un texto de más de 600 caracteres
# ══════════════════════════════════════════════════════════════════
lab.responder("M2.A1", "Escribe aquí tu análisis.")

In [ ]:
# ══════════════════════════════════════════════════════════════════
# RESPONDE · M2.A2 · Reingresos y visitados
#   FORMATO : Texto entre comillas de 40 a 600 caracteres, en oraciones completas. Usa los datos de TU caso.
#   EJEMPLO : lab.responder("M2.A2", "En mi red, la falla en ... deja ... clientes sin luz porque ...")
#   NO      : ""   "porque sí"   un texto de más de 600 caracteres
# ══════════════════════════════════════════════════════════════════
lab.responder("M2.A2", "Escribe aquí tu análisis.")

In [ ]:
# ══════════════════════════════════════════════════════════════════
# RESPONDE · M2.A3 · Tu tramo propuesto
#   FORMATO : Texto entre comillas de 40 a 600 caracteres, en oraciones completas. Usa los datos de TU caso.
#   EJEMPLO : lab.responder("M2.A3", "En mi red, la falla en ... deja ... clientes sin luz porque ...")
#   NO      : ""   "porque sí"   un texto de más de 600 caracteres
# ══════════════════════════════════════════════════════════════════
lab.responder("M2.A3", "Escribe aquí tu análisis.")

---
## Informe final (8 min)

Para cada escenario, elige la frontera: `"pila"`, `"cola"` o `"heap"`.

In [ ]:
# ══════════════════════════════════════════════════════════════════
# RESPONDE · INF.E1 · Escenario: ambulancia
#   FORMATO : Una de estas tres palabras, en minúscula y entre comillas: "pila", "cola" o "heap".
#   EJEMPLO : lab.responder("INF.E1", "cola")
#   NO      : "Pila"   "CustomPila"   "dijkstra"
# ══════════════════════════════════════════════════════════════════
lab.responder("INF.E1", "???")

In [ ]:
# ══════════════════════════════════════════════════════════════════
# RESPONDE · INF.E2 · Escenario: ¿sigue conectada?
#   FORMATO : Una de estas tres palabras, en minúscula y entre comillas: "pila", "cola" o "heap".
#   EJEMPLO : lab.responder("INF.E2", "heap")
#   NO      : "Pila"   "CustomPila"   "dijkstra"
# ══════════════════════════════════════════════════════════════════
lab.responder("INF.E2", "???")

In [ ]:
# ══════════════════════════════════════════════════════════════════
# RESPONDE · INF.E3 · Escenario: turista
#   FORMATO : Una de estas tres palabras, en minúscula y entre comillas: "pila", "cola" o "heap".
#   EJEMPLO : lab.responder("INF.E3", "pila")
#   NO      : "Pila"   "CustomPila"   "dijkstra"
# ══════════════════════════════════════════════════════════════════
lab.responder("INF.E3", "???")

In [ ]:
# ══════════════════════════════════════════════════════════════════
# RESPONDE · INF.A1 · Tu hallazgo principal
#   FORMATO : Texto entre comillas de 40 a 600 caracteres, en oraciones completas. Usa los datos de TU caso.
#   EJEMPLO : lab.responder("INF.A1", "En mi red, la falla en ... deja ... clientes sin luz porque ...")
#   NO      : ""   "porque sí"   un texto de más de 600 caracteres
# ══════════════════════════════════════════════════════════════════
lab.responder("INF.A1", "Escribe aquí tu hallazgo.")

In [ ]:
# Revisa qué te falta y entrega.
lab.estado()

In [ ]:
lab.entregar()

**¡Listo!** Puedes volver a ejecutar `lab.estado()` cuando quieras. Si te falta una revisión de
análisis, usa `lab.revisar("M1.A1")` (cambia el id).